# Pluie de demain: session 1 checkpoint

Fill in this notebook during the session, run it on Kaggle, then hand in the `.ipynb` file on Moodle
(*File → Download notebook* on Kaggle). No grade: it is a checkpoint for your team.

**Team name:** Master Shifu

**Members:**
- Gabriel DESTAL
- Georg RIEKHAKAINEN
- Chris KARAM
- Liam SZEFNER

## 1. Set up (20 min)

Replace `[ ]` with `[x]` when done.

- [ ] Team of 3 to 4 formed, team name sent to the teaching team.
- [ ] Every member has joined the Kaggle competition of track 1 (*How much?*), and the team is merged on Kaggle. Track 2 (*Will it rain?*) opens later.
- [ ] This notebook runs on Kaggle with the competition data attached (*Add Input* if needed).

In [ ]:
import os, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def find_data():
    hits = sorted(Path("/kaggle/input").glob("**/train.parquet"))
    if not hits:
        raise FileNotFoundError("train.parquet not found under /kaggle/input: add the competition data (Input panel)")
    return hits[0].parent

DATA = Path(os.environ.get("PDD_DATA") or find_data())
t0 = time.time()
train = pd.read_parquet(DATA / "train.parquet")
test = pd.read_parquet(DATA / "test.parquet")
maps = pd.read_parquet(DATA / "maps.parquet")
stations = pd.read_csv(DATA / "stations.csv", dtype={"station": str})
print(f"loaded in {time.time() - t0:.0f} s")

## 2. Getting hold of the data (40 min)

In [ ]:
#size of each file: rows, columns, memory, size on disk and load time
files = {"train": "train.parquet", "test": "test.parquet", "maps": "maps.parquet", "stations": "stations.csv"}
rows = []
for name, fname in files.items():
    t0 = time.time()
    df = pd.read_csv(DATA / fname, dtype={"station": str}) if fname.endswith(".csv") else pd.read_parquet(DATA / fname)
    rows.append({"file": fname, "rows": df.shape[0], "columns": df.shape[1],
                 "disk (MB)": (DATA / fname).stat().st_size / 1e6,
                 "memory (MB)": df.memory_usage(deep=True).sum() / 1e6,
                 "load (s)": time.time() - t0})
    del df
pd.DataFrame(rows).set_index("file").round(1)

In [ ]:
#what a row is, and how train and maps are joined
print("train: one row per (station, target_date), unique:", not train.duplicated(["station", "target_date"]).any())
print(f"  {train.station.nunique()} stations x {train.target_date.nunique()} days "
      f"({train.target_date.min():%Y-%m-%d} -> {train.target_date.max():%Y-%m-%d})")
print("maps: one row per target_date, unique:", maps.target_date.is_unique, f"({len(maps)} dates)")
groups = pd.Series([c.rsplit("_", 1)[0] for c in maps.columns[1:]]).value_counts()
print("  map variables x synoptic stations:", groups.to_dict())
print("  every train/test date has a map row:",
      train.target_date.isin(maps.target_date).all(), test.target_date.isin(maps.target_date).all())

#join train and maps for one station to see how many rows match
one = train[train.station == train.station.iloc[0]].merge(maps, on="target_date", how="left", validate="many_to_one")
print("  joined example:", one.shape)

#check the lags
s = train[train.station == train.station.iloc[0]].set_index("target_date")["rr"]
lagged = train[train.station == train.station.iloc[0]].set_index("target_date")
for k in (4, 10):
    match = np.isclose(lagged[f"rr_lag{k}"], s.shift(k, freq="D").reindex(lagged.index))
    print(f"rr_lag{k}(T) == rr(T-{k}): {np.nanmean(match):.1%} of days")

# check which day the map describes
obs = train.groupby("target_date")["rr"].mean()
map_rr = maps.set_index("target_date").filter(like="rr_day_").mean(axis=1)
corr = {f"T-{k}": map_rr.corr(obs.shift(k, freq="D").reindex(map_rr.index)) for k in range(0, 5)}
print("corr(map rain at row T, observed rain on day):", {k: round(float(v), 2) for k, v in corr.items()})

**What is a row of `train.parquet`? A row of `maps.parquet`? How are they joined?**

Answer: a row of `train.parquet` is one station on one day T (`id = station_YYYYMMDD`). It holds the target `rr`, the 24 h rainfall at that station on day T in mm, plus that station's past observations (`rr_lag4…10`, `tn/tx/ffm_lag4`) and its static metadata (lat, lon, alt, department, day of year, month). It has 6.9 M rows: 1,457 stations × 4,748 days (2010-01-01 to 2022-12-31). A row of `maps.parquet` is one day T for all of France. It holds 9 synoptic variables (`pmer`, `dp24`, `t`, `td`, `u`, `uw`, `vw`, `n`, `rr_day`) measured at 37 synoptic stations (WMO ids `07xxx`), so 333 columns. They are joined many-to-one on `target_date`: every station on day T gets the same map row.

**Which dates do a row's features come from, and why?**

Answer: for a target day T, the station features come from T−10 to T−4 (rain on each of those 7 days, temperature and wind on T−4 only). The map comes from **T−2**. The checks above confirm both: `rr_lagk(T)` equals the station's `rr` on day T−k, and the map's rain on row T correlates best with observed rain on T−2 (r ≈ 0.87, against 0.36 for T−1). The features only use what a forecaster would actually have when issuing the forecast. Daily records from the dense station network arrive with a delay of a few days (hence T−4 at best). The synoptic network is sent in near real time (hence T−2). Using anything closer to T would leak information that is unavailable in operation and inflate the validation score.

In [ ]:
#stations of test.parquet absent from train.parquet
new = sorted(set(test.station) - set(train.station))
print(f"test stations: {test.station.nunique()}, of which {len(new)} never appear in train "
      f"({test.station.isin(new).mean():.1%} of test rows)")
print("all test stations are described in stations.csv:", set(test.station) <= set(stations.station))

new_st = stations[stations.station.isin(new)]
by_dep = new_st.groupby("dep").agg(n_stations=("station", "size"), alt_median=("alt", "median"))
by_dep["dep_in_train"] = by_dep.index.astype(str).str.zfill(2).isin(train.dep.unique())
print(by_dep)

#test dates
print(f"test dates: {test.target_date.nunique()} days from {test.target_date.min():%Y-%m-%d} to {test.target_date.max():%Y-%m-%d}, "
      f"typical gap {test.groupby('station').target_date.diff().mode()[0].days} days")

fig, ax = plt.subplots(figsize=(6, 6))
seen = stations[~stations.station.isin(new)]
ax.scatter(seen.lon, seen.lat, s=4, c="lightgrey", label="in train")
ax.scatter(new_st.lon, new_st.lat, s=6, c="crimson", label="test only")
ax.set(xlabel="longitude", ylabel="latitude", title="Test stations absent from train")
ax.set_aspect(1 / np.cos(np.radians(46)))
ax.legend()
plt.show()

**What did you find about the stations of the test set?**

Answer: the test set covers all 1,660 stations in `stations.csv`. 203 of them (12% of test rows) never appear in train. They are not scattered at random. They make up 7 entire departments that are missing from train: Ardèche (07), Aude (11), Corsica (20), Gard (30), Hérault (34), Lozère (48) and Pyrénées-Orientales (66). That is essentially the Mediterranean rim, the Cévennes and Corsica, the area of France with the heaviest and most intense autumn rain events ("épisodes cévenols"). So the test is a split in time (90 dates, one every 12 days, 2023-01-20 to 2025-12-23, all after the 2022 end of train) and in space. A model that relies on a station's identity or its learned climatology will have nothing for those stations. It must generalise from lat/lon/alt, the station's own lags and the map. Our validation should mimic this by holding out whole departments, not just later dates.

## 3. A first exploratory analysis (60 min)

Four figures, each followed by one sentence of conclusion.

In [ ]:
# Figure 1. The target: share of dry days (RR = 0), share of days under 1 mm, distribution of RR on rainy days (log scale)


**Conclusion:** …

In [ ]:
# Figure 2. Seasonality: frequency of rainy days by month


**Conclusion:** …

In [ ]:
# Figure 3. Space: frequency of rainy days by station, against altitude or on a map (latitude/longitude)


**Conclusion:** …

In [ ]:
# Figure 4. Missing values: share of missing values per column of train.parquet, and of the map over time


**Conclusion:** …

## 4. Stepping back on the starter (40 min)

Short answers, a few lines each.

**1. Which validation does it use, and why must days never be shuffled?**

Answer: …

**2. Why combine the two models as exp(p·m1 + (1 − p)·m0) − 1 for the RMSLE, rather than p × predicted amount?**

Answer: …

**3. Its PCA keeps 20 components: is that justified? What does the median imputation do when map columns are empty?**

Answer: …

**4. How does it predict a station it has never seen?**

Answer: …

**5. Is its Brier score good? Compared with what?**

Answer: …

## 5. Plan (20 min)

- [ ] A first submission in track 1 (the starter's `submission.csv` is fine). Public score obtained: …

**Three things to try next, and who does what:**

1. … (who: …)
2. … (who: …)
3. … (who: …)